# Scientific Construction Risk Modeling\nStreamlit + Machine Learning + SHAP analysis

In [ ]:
# ==========================================================# НАУЧНАЯ ВЕРСИЯ — МОДЕЛИРОВАНИЕ СТРОИТЕЛЬНЫХ РИСКОВ# ==========================================================import ioimport mathimport numpy as npimport pandas as pdimport streamlit as stimport matplotlib.pyplot as pltimport seaborn as snsimport shapfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegression, Ridgefrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import (    mean_squared_error,    mean_absolute_error,    r2_score,    accuracy_score,    confusion_matrix)from sklearn.inspection import permutation_importance# ==========================================================# CONFIG# ==========================================================st.set_page_config(    page_title="Анализ строительных рисков",    layout="wide")st.title("Научное моделирование строительных рисков")tabs = st.tabs([    "Моделирование риска",    "Симуляция UCB",    "ℹ О проекте"])# ==========================================================# TAB 1 — MODELING# ==========================================================with tabs[0]:    st.header("Интеллектуальное прогнозирование риска")    uploaded_file = st.file_uploader(        "Загрузите Excel-файл (.xlsx)",        type=["xlsx"]    )    if not uploaded_file:        st.info("Загрузите файл для анализа.")        st.stop()    raw_df = pd.read_excel(uploaded_file)    raw_df.columns = raw_df.iloc[0]    df = raw_df.drop(index=0).reset_index(drop=True)    df = df.loc[:, ~df.columns.str.contains("ИТОГО", case=False)]    df.columns = df.columns.astype(str).str.strip()    df = df.apply(pd.to_numeric)    target_column = "Индекс риска(%)"    if target_column not in df.columns:        st.error("Не найден столбец 'Индекс риска(%)'")        st.stop()    df = df.rename(columns={target_column: "Индекс_качества"})    st.success(f"Данные успешно загружены. Наблюдений: {len(df)}")    st.subheader("Предварительный просмотр данных")    col1, col2 = st.columns(2)    with col1:        st.write("Первые строки датасета")        st.dataframe(df.head())    with col2:        st.write("Статистическое описание")        st.dataframe(df.describe())    st.write("Размер датасета:", df.shape)    features = [c for c in df.columns if c != "Индекс_качества"]    target = "Индекс_качества"    X = df[features]    y = df[target]    st.write(f"Количество факторов: {len(features)}")    X_train, X_test, y_train, y_test = train_test_split(        X, y,        test_size=0.2,        random_state=42    )    models = {        "Linear Regression": Pipeline([            ("scaler", StandardScaler()),            ("model", LinearRegression())        ]),        "Ridge Regression": Pipeline([            ("scaler", StandardScaler()),            ("model", Ridge(alpha=1.0))        ]),        "Random Forest": RandomForestRegressor(            n_estimators=300,            random_state=42,            n_jobs=-1        )    }    results_summary = []    for name, model in models.items():        model.fit(X_train, y_train)        y_pred = model.predict(X_test)        mse = mean_squared_error(y_test, y_pred)        rmse = np.sqrt(mse)        mae = mean_absolute_error(y_test, y_pred)        r2 = r2_score(y_test, y_pred)        cv_scores = cross_val_score(            model,            X,            y,            cv=5,            scoring="neg_mean_squared_error"        )        results_summary.append({            "Модель": name,            "MSE": mse,            "RMSE": rmse,            "MAE": mae,            "R2": r2,            "CV_MSE_mean": -cv_scores.mean()        })    df_models = pd.DataFrame(results_summary).sort_values("RMSE")    st.subheader("Сравнение моделей")    st.dataframe(df_models)    best_model_name = df_models.iloc[0]["Модель"]    st.success(f"Лучшая модель: {best_model_name}")    best_model = models[best_model_name]    best_model.fit(X_train, y_train)    y_pred = best_model.predict(X_test)    st.markdown(f"""    ### Метрики лучшей модели    - RMSE: {df_models.iloc[0]['RMSE']:.4f}    - MAE: {df_models.iloc[0]['MAE']:.4f}    - R²: {df_models.iloc[0]['R2']:.4f}    - CV MSE: {df_models.iloc[0]['CV_MSE_mean']:.4f}    """)    st.subheader("Реальный риск vs Предсказанный риск")    fig_risk, ax = plt.subplots(figsize=(7,6))    sns.scatterplot(x=y_test, y=y_pred, ax=ax)    ax.plot(        [y_test.min(), y_test.max()],        [y_test.min(), y_test.max()],        color="red",        linestyle="--"    )    ax.set_xlabel("Реальный риск")    ax.set_ylabel("Предсказанный риск")    st.pyplot(fig_risk)    st.subheader("Анализ остатков")    residuals = y_test - y_pred    fig_res, ax = plt.subplots(figsize=(7,5))    sns.scatterplot(x=y_pred, y=residuals, ax=ax)    ax.axhline(0, linestyle="--", color="red")    ax.set_xlabel("Предсказанные значения")    ax.set_ylabel("Остатки")    st.pyplot(fig_res)    perm = permutation_importance(        best_model,        X_test,        y_test,        n_repeats=10,        random_state=42,        scoring="neg_mean_squared_error"    )    factor_importance = pd.DataFrame({        "Фактор": features,        "Влияние_на_MSE": -perm.importances_mean    }).sort_values("Влияние_на_MSE", ascending=False)    st.subheader("Permutation Importance")    st.dataframe(factor_importance)    fig_imp, ax = plt.subplots(figsize=(8, 6))    sns.barplot(        data=factor_importance,        x="Влияние_на_MSE",        y="Фактор",        ax=ax    )    st.pyplot(fig_imp)    st.subheader("Распределение факторов")    selected_feature = st.selectbox(        "Выберите фактор",        features    )    fig_dist, ax = plt.subplots(figsize=(7,5))    sns.histplot(        df[selected_feature],        kde=True,        ax=ax    )    st.pyplot(fig_dist)    st.subheader("SHAP-анализ")    if best_model_name == "Random Forest":        explainer = shap.TreeExplainer(best_model)        shap_values = explainer(X_test)        shap_array = shap_values.values        fig = plt.figure()        shap.summary_plot(shap_values, X_test, show=False)        st.pyplot(fig)    else:        scaler = best_model.named_steps["scaler"]        linear_model = best_model.named_steps["model"]        X_train_scaled = scaler.transform(X_train)        X_test_scaled = scaler.transform(X_test)        explainer = shap.LinearExplainer(            linear_model,            X_train_scaled,            feature_perturbation="interventional"        )        shap_array = explainer.shap_values(X_test_scaled)        fig = plt.figure()        shap.summary_plot(            shap_array,            X_test_scaled,            feature_names=X.columns,            show=False        )        st.pyplot(fig)    low = y.quantile(0.35)    high = y.quantile(0.65)    def classify(v):        if v < low:            return "Критический"        elif v < high:            return "Средний"        else:            return "Отличный"    df_res = X_test.copy()    df_res["Реальный риск"] = y_test    df_res["Предсказанный риск"] = y_pred    df_res["Реальный класс"] = df_res["Реальный риск"].apply(classify)    df_res["Предсказанный класс"] = df_res["Предсказанный риск"].apply(classify)    acc = accuracy_score(        df_res["Реальный класс"],        df_res["Предсказанный класс"]    )    st.metric("Accuracy", f"{acc*100:.2f}%")    st.subheader("Матрица ошибок")    cm = confusion_matrix(        df_res["Реальный класс"],        df_res["Предсказанный класс"],        labels=["Критический","Средний","Отличный"]    )    fig_cm, ax = plt.subplots(figsize=(6,5))    sns.heatmap(        cm,        annot=True,        fmt="d",        cmap="Blues",        xticklabels=["Критический","Средний","Отличный"],        yticklabels=["Критический","Средний","Отличный"],        ax=ax    )    ax.set_xlabel("Предсказанный класс")    ax.set_ylabel("Реальный класс")    st.pyplot(fig_cm)